In [21]:
import fitz
import chromadb
import requests

from sentence_transformers import SentenceTransformer

In [22]:
doc = fitz.open("data/notes.pdf")

print("Total Pages:", len(doc))

Total Pages: 140


In [23]:
pages_data = []

for page_num in range(len(doc)):

    page = doc[page_num]

    text = page.get_text()

    pages_data.append({
        "page": page_num + 1,
        "text": text
    })

print("Pages Extracted:", len(pages_data))

Pages Extracted: 140


In [24]:
print("Page Number:", pages_data[0]["page"])

print("\nText Preview:\n")

print(pages_data[0]["text"][:500])

Page Number: 1

Text Preview:

Machine Learning Handbook
Predrag Radivojac and Martha White
November 5, 2019



In [25]:
chunk_size = 1000

chunks_with_metadata = []

for page_data in pages_data:

    page_num = page_data["page"]

    text = page_data["text"]

    for i in range(0, len(text), chunk_size):

        chunk = text[i:i + chunk_size]

        chunks_with_metadata.append({
            "text": chunk,
            "page": page_num,
            "filename": "notes.pdf"
        })

print("Total Chunks:", len(chunks_with_metadata))

Total Chunks: 376


In [26]:
print(chunks_with_metadata[0])

{'text': 'Machine Learning Handbook\nPredrag Radivojac and Martha White\nNovember 5, 2019\n', 'page': 1, 'filename': 'notes.pdf'}


In [27]:
texts = [item["text"] for item in chunks_with_metadata]

print("Total Text Chunks:", len(texts))

Total Text Chunks: 376


In [28]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

chunk_embeddings = model.encode(texts)

print(chunk_embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(376, 384)


In [42]:
import chromadb

client = chromadb.PersistentClient(
    path="./chroma_db"
)

collection_metadata = client.get_or_create_collection(
    name="ml_handbook_metadata"
)


print("Collection Created")

Collection Created


In [31]:
ids = []
documents = []
embeddings = []
metadatas = []

for i, item in enumerate(chunks_with_metadata):

    ids.append(f"chunk_{i}")

    documents.append(item["text"])

    embeddings.append(chunk_embeddings[i].tolist())

    metadatas.append({
        "page": item["page"],
        "filename": item["filename"]
    })

print(len(ids))
print(len(documents))
print(len(embeddings))
print(len(metadatas))

376
376
376
376


In [32]:
collection_metadata.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)

print("All chunks stored successfully!")

All chunks stored successfully!


In [41]:
print("Total Chunks Stored:", collection_metadata.count())

Total Chunks Stored: 376


In [34]:
query = "What is probability theory?"

query_embedding = model.encode(query)

results = collection_metadata.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

In [35]:
print(results["metadatas"][0])

[{'page': 13, 'filename': 'notes.pdf'}, {'filename': 'notes.pdf', 'page': 11}, {'filename': 'notes.pdf', 'page': 12}]


In [39]:
def ask_rag_with_sources(question):

    # Create query embedding
    query_embedding = model.encode(question)

    # Retrieve chunks
    results = collection_metadata.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=3
    )

    retrieved_chunks = results["documents"][0]
    retrieved_metadata = results["metadatas"][0]

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
    Answer the question using only the provided context.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2",
            "prompt": prompt,
            "stream": False
        }
    )

    answer = response.json()["response"]

    sources = []

    for meta in retrieved_metadata:
        sources.append(
            f"{meta['filename']} (Page {meta['page']})"
        )

    return answer, sources

In [40]:
ask_rag_with_sources(
    "What is probability theory?"
)

('Probability theory is a branch of mathematics that deals with measuring the likelihood of events.',
 ['notes.pdf (Page 13)', 'notes.pdf (Page 11)', 'notes.pdf (Page 12)'])